<a href="https://colab.research.google.com/github/mahendrapd/GRST/blob/main/InfiltrationRST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [101]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [102]:
data = pd.read_parquet('/content/drive/MyDrive/dataset/training.parquet')

In [103]:
cols=len(data.columns)
nfeatures = cols-1
X = data.iloc[:,0:nfeatures]
y = data.iloc[:,-1]
print(len(data),cols)

347122 55


In [104]:
targetclass='Infiltration'

In [105]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [106]:
def RSTClassifier(X,y,targetclass=y[0]):
    #print(targetclass)
    dc=pd.Series(y)
    cm=dc.groupby(dc).apply(lambda px: px.index.tolist()).to_dict() #index of targetclass
    D=set(cm[targetclass])
    # D1=set(cm['Bruteforce'])
    # D2=set(cm['Botnet'])
    # D3=set(cm['Portscan'])
    # D4=set(cm['Webattack'])
    # D5=set(cm['DoS'])
    # D6=set(cm['Infiltration'])
    # D7=set(cm['DDoS'])
    print(D)
    nfeatures=len(X.columns)
    #print(cm)
    purity={}
    headers = X.keys().tolist()

    for j in range(nfeatures):
        header=headers[j]
        hdata={}
        #print("Columns:",j,header)
        dp=pd.Series(X.iloc[:,j])
        md=dp.groupby(dp).apply(lambda px: px.index.tolist()).to_dict()
        K=md.keys()
        #print(K)
        for z in K:
            common_elements=set(md[z]) & D
            val=round(len(common_elements)/len(md[z]),3)
            #print(z,len(md[z]),len(common_elements),val)
            hdata[z]=val
        purity[header]=hdata
    #print(purity['urgent'][0])
    return purity

In [107]:
pur=RSTClassifier(X_train,y_train,targetclass)

{262147, 262148, 262149, 262150, 262152, 262153, 262154, 262156, 262158, 262159, 262160, 262161, 262162, 262163, 262164, 262165, 262166, 262168, 262169, 262170, 262171, 262173, 262174, 262175, 262176, 262177, 262178, 262180, 262182, 262184, 262185, 262186, 262187, 262188, 262189, 262191, 262192, 262193, 262194, 262195, 262196, 262197, 262198, 262199, 262332, 262333, 262334, 262337, 262338, 262342, 262344, 262347, 262348, 262352, 262353, 262354, 262355, 262356, 262358, 262359, 262361, 262362, 262363, 262364, 262365, 262368, 262369, 262371, 262373, 262374, 262375, 262377, 262382, 262383, 262384, 262385, 262386, 262388, 262390, 262392, 262393, 262394, 262397, 262400, 262401, 262402, 262404, 262405, 262406, 262407, 262408, 262410, 262411, 262413, 262414, 262415, 262416, 262418, 262419, 262422, 262423, 262424, 262425, 262426, 262427, 262428, 262462, 262463, 262464, 262465, 262467, 262468, 262469, 262470, 262472, 262473, 262474, 262475, 262476, 262477, 262479, 262480, 262481, 262482, 262483,

In [108]:
#print(pur)

In [109]:
def RST_AmbigiousSample(X,pur):
    headers = X.keys().tolist()
    print(headers,len(headers),len(X))
    ambigioussamples={}
    for i in range(len(X)):
        flag=0
        sumpurity=0
        for j in range(len(headers)):
            key=X.iloc[i][headers[j]]
            if (pur[headers[j]][key] == 1 or pur[headers[j]][key] == 0):
                flag=1
                break
            else:
                sumpurity+=pur[headers[j]][key]
        if (flag == 0):
            Avgpurity=round(sumpurity/len(headers),3)
            ambigioussamples[i]=Avgpurity
    return ambigioussamples

In [110]:
AmbSample=RST_AmbigiousSample(X_train,pur)

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle 

In [111]:
print(len(AmbSample))

178459


In [112]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [113]:
def RST_Gradient(y,sample,epoch=300):
    Theta=0.5
    LRate=0.01
    pc=0
    for i in range(epoch):
        correctcount=0
        val=0
        Th=Theta
        L=0

        for key, value in sample.items():
            if (value >= Theta and y.iloc[key] == targetclass):
                correctcount+=1
            elif (value < Theta and y.iloc[key] != targetclass):
                correctcount+=1
            else:
                val+=value
        ictotal=(len(sample)-correctcount)
        try:
          p=correctcount/len(sample)
          #L=-(Th*math.log2(p)+(1-Th)*math.log2(1-p))
          #L=-((Th/p)-((1-Th)/(1-p)))
          #L=np.tanh(L)
          #Theta=abs(round((Theta-LRate*L),3))
          #print(L)
          if (pc<=correctcount):
            Theta=abs(round(Theta-LRate*(sigmoid((val/ictotal))),3))
            pc=correctcount
          #Theta=round(Theta-LRate*(0-(val/ictotal))*(val/ictotal)*(1-(val/ictotal)),3)
        except:
          Theta=Theta
        print(i, correctcount, ictotal, len(sample), val, Theta)

    return Theta

In [114]:
Th=RST_Gradient(y_train,AmbSample,300)

0 171656 6803 178459 193.79799999999372 0.495
1 171656 6803 178459 193.79799999999372 0.49
2 171656 6803 178459 193.79799999999372 0.485
3 171656 6803 178459 193.79799999999372 0.48
4 171656 6803 178459 193.79799999999372 0.475
5 171656 6803 178459 193.79799999999372 0.47
6 171656 6803 178459 193.79799999999372 0.465
7 171656 6803 178459 193.79799999999372 0.46
8 171656 6803 178459 193.79799999999372 0.455
9 171656 6803 178459 193.79799999999372 0.45
10 171656 6803 178459 193.79799999999372 0.445
11 171656 6803 178459 193.79799999999372 0.44
12 171656 6803 178459 193.79799999999372 0.435
13 171656 6803 178459 193.79799999999372 0.43
14 171656 6803 178459 193.79799999999372 0.425
15 171656 6803 178459 193.79799999999372 0.42
16 171656 6803 178459 193.79799999999372 0.415
17 171656 6803 178459 193.79799999999372 0.41
18 171656 6803 178459 193.79799999999372 0.405
19 171656 6803 178459 193.79799999999372 0.4
20 171656 6803 178459 193.79799999999372 0.395
21 171656 6803 178459 193.79799999

In [ ]:
def RST_Validation(X,y,pur,Theta):
    TP=0
    TN=0
    FP=0
    FN=0
    headers = X.keys().tolist()
    for i in range(len(X)):
        val=0
        count=0
        pval=0

        flag=0
        for j in range(len(headers)):
            #KYS=list(pur[headers[j]].keys())
            #dflag=0
            key=X.iloc[i][headers[j]]
            try:
              pval=pur[headers[j]][key]
            except KeyError:
              pval=Theta

            if ((pval == 1 or pval == 0)):
                flag=1
                break
            else:
                val+=pval
                count+=1

        if (flag == 1):
            if (pval==1 and y.iloc[i] == targetclass):
                TP+=1
            elif (pval==1 and y.iloc[i] != targetclass):
                FN+=1
            elif (pval==0 and y.iloc[i] != targetclass):
                TN+=1
            else:
                FP+=1
        else:
            Aval=round((val/count),3)
            if (Aval>=Theta and y.iloc[i] == targetclass):
                TP+=1
            elif (Aval<Theta and y.iloc[i] != targetclass):
                TN+=1
            elif (Aval>=Theta and y.iloc[i] != targetclass):
                FN+=1
            else:
                FP+=1
    try:
      Recall=round(TP/(TP+FN),4)
    except:
      Recall=0
    try:
      FPR=round(FP/(TN+FP),4)
    except:
      FPR=0
    try:
      Precision=round(TP/(TP+FP),4)
    except:
      Precision=0
    try:
      Accuracy=round((TP+TN)/(TP+TN+FP+FN),4)
    except:
      Accuracy=0
    try:
      FScore=round(2*Recall*Precision/(Recall+Precision),4)
    except:
      FScore=0
    print("TP=",TP,"\tTN=",TN,"\tFP=",FP,"\tFN=",FN)
    return Recall, FPR, Precision, FScore, Accuracy

In [116]:
Result=RST_Validation(X_train,y_train,pur,Th)
print(Result)

TP= 5 	TN= 253524 	FP= 6809 	FN= 3
(0.625, 0.0262, 0.0007, 0.0014, 0.9738)


In [117]:
ResultTest = RST_Validation(X_test,y_test,pur,Th)
print(ResultTest)

TP= 0 	TN= 84535 	FP= 2243 	FN= 3
(0.0, 0.0258, 0.0, 0, 0.9741)


In [ ]:
def CountPatterns(pur):
  value=0
  for outer_key, outer_value in pur.items():
    acount=0
    ncount=0
    for inner_key, inner_value in outer_value.items():
      if (inner_value == 0):
        acount=acount+1
      if (inner_value == 1):
        ncount=ncount+1
    print(f"{outer_key}: {acount}, {ncount}, {len(outer_value.items())}")

In [119]:
CountPatterns(pur)

Flow Duration: 12, 0, 13
Total Fwd Packets: 68, 0, 72
Total Backward Packets: 46, 0, 50
Fwd Packets Length Total: 6, 0, 12
Bwd Packets Length Total: 34, 0, 37
Fwd Packet Length Max: 42, 0, 45
Fwd Packet Length Mean: 36, 0, 46
Fwd Packet Length Std: 36, 0, 43
Bwd Packet Length Max: 28, 0, 32
Bwd Packet Length Mean: 11, 0, 16
Bwd Packet Length Std: 26, 0, 32
Flow Bytes/s: 85, 0, 89
Flow Packets/s: 21, 0, 37
Flow IAT Mean: 4, 0, 5
Flow IAT Std: 11, 0, 12
Flow IAT Max: 13, 0, 14
Flow IAT Min: 9, 0, 10
Fwd IAT Total: 12, 0, 13
Fwd IAT Mean: 4, 0, 5
Fwd IAT Std: 11, 0, 12
Fwd IAT Max: 13, 0, 14
Fwd IAT Min: 9, 0, 10
Bwd IAT Total: 0, 0, 101
Bwd IAT Mean: 40, 0, 101
Bwd IAT Std: 46, 0, 101
Bwd IAT Max: 20, 0, 101
Bwd IAT Min: 50, 0, 101
Fwd Header Length: 88, 0, 89
Bwd Header Length: 7, 0, 8
Fwd Packets/s: 28, 0, 46
Bwd Packets/s: 22, 0, 34
Packet Length Max: 45, 0, 49
Packet Length Mean: 14, 0, 23
Packet Length Std: 28, 0, 32
Packet Length Variance: 12, 0, 13
Avg Packet Size: 15, 0, 24
Avg F